Import libraries/functions

In [37]:
import pandas as pd
import geopandas as gpd
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.seasonal import STL
import matplotlib.pyplot as plt

Load the shapefile to filter down to only LA area codes

In [2]:
shapefile_path = "../../data/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2)/Local_Authority_Districts_(May_2025)_Boundaries_UK_BFC_(V2).shp"  # Change to your shapefile path
id_column = "LAD25CD"       # Column where first letter is E/W/S/N
name_column = "LAD25NM"   # Column with local authority names

# --- Load shapefile ---
la_list = gpd.read_file(shapefile_path)

In [3]:
la_list = la_list[['LAD25CD']]

Load the data

In [31]:
hpi_raw = pd.read_csv(filepath_or_buffer="../../data/UK-HPI-full-file-2025-05.csv")
hpi_raw = hpi_raw[hpi_raw['AreaCode'].isin(la_list['LAD25CD'])]
hpi_raw = hpi_raw[['Date', 'RegionName', 'AreaCode', 'AveragePrice']]
### Filter for England Wales (area code in hpi index identifies England and Wales by starting the area code with E or W. K stands for UK)
hpi_raw_ew = hpi_raw[hpi_raw["AreaCode"].str.startswith(("E", "W"))]
hpi_raw_ew['Date'] = pd.to_datetime(hpi_raw_ew['Date'], format='%d/%m/%Y')

C:\Users\slong\AppData\Local\Temp\ipykernel_16860\2911113785.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  hpi_raw_ew['Date'] = pd.to_datetime(hpi_raw_ew['Date'], format='%d/%m/%Y')


In [10]:
### Check that only england and wales are here
print(hpi_raw_ew.AreaCode.str[0].unique())

['E' 'W']


In [32]:
hpi_raw_ew

,Date,RegionName,AreaCode,AveragePrice
257,1995-01-01,Adur,E07000223,54669
258,1995-02-01,Adur,E07000223,55864
259,1995-03-01,Adur,E07000223,55880
260,1995-04-01,Adur,E07000223,55596
261,1995-05-01,Adur,E07000223,53483
...,...,...,...,...
145559,2025-01-01,York,E06000014,305903
145560,2025-02-01,York,E06000014,304052
145561,2025-03-01,York,E06000014,305832
145562,2025-04-01,York,E06000014,307638


In [35]:
hpi_raw_ew_ts = []

# Monthly frequency
freq = 'MS'

for area in hpi_raw_ew['AreaCode'].unique():
    area_df = hpi_raw_ew[hpi_raw_ew['AreaCode'] == area].copy()
    
    # Set date index and ensure proper frequency
    area_df = area_df.set_index('Date').sort_index().asfreq(freq)

    area_df[['RegionName', 'AreaCode']] = area_df[['RegionName', 'AreaCode']].ffill()

    
    # Optionally interpolate missing values
    area_df['AveragePrice'] = area_df['AveragePrice'].interpolate()
    
    # Perform decomposition
    stl = STL(area_df['AveragePrice'], period=12)
    decomposition = stl.fit()
    
    # Add components back to the DataFrame
    area_df['trend'] = decomposition.trend
    area_df['seasonal'] = decomposition.seasonal
    area_df['residual'] = decomposition.resid

    # Reset index so date is a column again
    area_df = area_df.reset_index()
    
    # Store for concatenation
    hpi_raw_ew_ts.append(area_df)

# Combine all area_code DataFrames into one
final_df = pd.concat(hpi_raw_ew_ts, ignore_index=True)

In [36]:
final_df

,Date,RegionName,AreaCode,AveragePrice,trend,seasonal,residual
0,1995-01-01,Adur,E07000223,54669,55947.509070,-1144.426324,-134.082745
1,1995-02-01,Adur,E07000223,55864,55608.485886,-1054.927121,1310.441234
2,1995-03-01,Adur,E07000223,55880,55280.130671,-405.871290,1005.740620
3,1995-04-01,Adur,E07000223,55596,54963.627142,-225.564410,857.937268
4,1995-05-01,Adur,E07000223,53483,54659.459960,-780.334545,-396.125415
...,...,...,...,...,...,...,...
115700,2025-01-01,York,E06000014,305903,309840.340877,-4101.361592,164.020715
115701,2025-02-01,York,E06000014,304052,310074.045456,-4444.799899,-1577.245557
115702,2025-03-01,York,E06000014,305832,310303.215905,-2301.822616,-2169.393289
115703,2025-04-01,York,E06000014,307638,310528.239840,-2503.035592,-387.204248


In [33]:
area_df = hpi_raw_ew[hpi_raw_ew['AreaCode'] == 'E07000223'].copy()
    
    # Set date index and ensure proper frequency
area_df = area_df.set_index('Date').sort_index().asfreq(freq)

area_df[['RegionName', 'AreaCode']] = area_df[['RegionName', 'AreaCode']].ffill()

# Optionally interpolate missing values
# area_df['AveragePrice'] = area_df['AveragePrice'].interpolate()

# # Perform decomposition
# stl = STL(area_df['AveragePrice'], period=12)
# decomposition = stl.fit()

# # Add components back to the DataFrame
# area_df['trend'] = decomposition.trend
# area_df['seasonal'] = decomposition.seasonal
# area_df['residual'] = decomposition.resid
# area_df['AreaCode'] = area  # Re-add if needed

# # Reset index so date is a column again
# area_df = area_df.reset_index()

In [34]:
area_df

,RegionName,AreaCode,AveragePrice
Date,,,
1995-01-01,Adur,E07000223,54669
1995-02-01,Adur,E07000223,55864
1995-03-01,Adur,E07000223,55880
1995-04-01,Adur,E07000223,55596
1995-05-01,Adur,E07000223,53483
...,...,...,...
2025-01-01,Adur,E07000223,377343
2025-02-01,Adur,E07000223,368874
2025-03-01,Adur,E07000223,365644
